# RMT Assignment 4 — test-set evaluation (Participant 2 / 3)

Runs `src/eval.py` on Kaggle: downloads each checkpoint from the public Hub repo, evaluates
MMLU (1,260 + 450), GSM8K (500), MedMCQA (500), CaseHOLD (500) and MBPP (500) with lm-eval 0.4.13,
and saves the answer to every test item in `results/eval/<run_id>/<checkpoint>/`.

**Before running**
1. *Settings → Accelerator*: **GPU T4 x2** — evaluation runs two workers, one per GPU (timing comparability only matters for training).
2. *Settings → Internet*: **On** (phone must be verified on Kaggle).
3. Optional: *Add-ons → Secrets* → `HF_TOKEN` (write token) — then every finished checkpoint is also uploaded to `PUSH_REPO`, so nothing is lost if the session dies.
4. First run with `MODE = "pilot"` (≈10 min), check Stage 0 numbers, then `MODE = "full"` → **Save Version → Save & Run All**.
5. When finished: *Output* → `eval_results.zip` → send to the team / commit `results/eval/` to GitHub.

The test set is used only for final numbers — never change settings after looking at it.

In [ ]:
import os, subprocess, sys, time, glob
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# ===== CONFIG =====
REPO_URL  = "https://github.com/alishermede/parameter-efficient-fine-tuning-and-catastrophic-forgetting-in-small-language-models.git"
HUB_REPO  = "alishermede/rmt-peft-ckpts"   # where the checkpoints are (public)
PUSH_REPO = ""                              # e.g. "<your-hf-login>/rmt-peft-ckpts" (needs HF_TOKEN secret); "" = no upload
MODE      = "pilot"                         # "pilot" (Stage 0, 50 items per task) | "full"

# Which checkpoints (MODE == "full"). Participant 2: full + lora of the 9 primary runs, plus Stage 0.
PLAN_ARGS = "--methods full,lora --stage0 --name participant2"
# Participant 3 examples:
#   PLAN_ARGS = "--methods adapter --name participant3_adapter"
#   PLAN_ARGS = "--order CLM --methods full,lora,adapter --seeds 1 --name reversed"
#   PLAN_ARGS = "--model replication --methods full,lora,adapter --seeds 1 --stage0 --name replication"
#   (replication / reversed checkpoints may live in another HF repo -> set HUB_REPO accordingly)
N_GPUS = 2

In [ ]:
!git clone --depth 1 $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao
!git log -1 --format="commit %H  %s"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("no HF_TOKEN secret (fine for downloading a public repo):", type(e).__name__)
    if PUSH_REPO:
        raise SystemExit("PUSH_REPO is set but there is no HF_TOKEN secret")
os.environ["HF_ALLOW_CODE_EVAL"] = "1"
os.environ["CKPT_CACHE"] = "/tmp/ckpt_cache"   # outside /kaggle/working (20 GB output limit)
os.makedirs(os.environ["CKPT_CACHE"], exist_ok=True)

In [ ]:
# Rebuild data/test/ (not in git) and verify against data/SHA256SUMS - must print 'verified'
!python src/prepare_data.py

In [ ]:
PUSH = f"--push_repo {PUSH_REPO}" if PUSH_REPO else ""
if MODE == "pilot":
    # Stage 0 on 50 items per task (MMLU: 2 per subject). Never reported. Expect MMLU roughly 40-50%,
    # GSM8K roughly 30-40% for Qwen2.5-0.5B; NaN / ~25% everywhere means an fp16 or prompt problem.
    !python src/eval.py --run_id q25-05b_full_MLC_s1 --checkpoint stage0 --pilot 50 --hub_repo {HUB_REPO} --overwrite
    !cat results/eval_pilot/q25-05b_base/stage0/summary.json | head -40
elif MODE == "full":
    !python src/plan_eval.py {PLAN_ARGS} --gpus {N_GPUS}
    name = PLAN_ARGS.split("--name")[1].split()[0]
    files = sorted(glob.glob(f"results/eval/jobs/{name}_gpu*.txt")) if N_GPUS > 1 else [f"results/eval/jobs/{name}.txt"]
    procs = []
    for gpu, jf in enumerate(files):
        log = open(f"/kaggle/working/eval_gpu{gpu}.log" if os.path.isdir("/kaggle") else f"eval_gpu{gpu}.log", "w")
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
        cmd = [sys.executable, "src/eval.py", "--jobs", jf, "--hub_repo", HUB_REPO, "--keep_going"] + PUSH.split()
        print("GPU", gpu, ":", open(jf).read().count("\n"), "jobs ->", log.name)
        procs.append(subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT))
    t0 = time.time()
    while any(p.poll() is None for p in procs):
        time.sleep(300)
        done = len(glob.glob("results/eval/*/*/summary.json"))
        print(f"{(time.time()-t0)/60:6.0f} min  checkpoints with summary.json: {done}", flush=True)
    print("exit codes:", [p.returncode for p in procs])
else:
    raise ValueError(MODE)

In [ ]:
# Per-checkpoint scores of this session
import json
for f in sorted(glob.glob("results/eval*/*/*/summary.json")):
    s = json.load(open(f))
    sc = s.get("scores", {})
    print(f"{s.get('run_id','?'):26s} {s.get('checkpoint','?'):20s} " +
          "  ".join(f"{k}={100*v['score']:.1f}" for k, v in sc.items()) +
          f"  ({s.get('total_seconds', 0)/60:.0f} min)")
!tail -n 30 /kaggle/working/eval_gpu*.log 2>/dev/null | grep -E "FAILED|Error|all jobs done" || true

In [ ]:
# Downloadable copy (Output tab): per-item answers, lm-eval outputs, summaries
!cd /kaggle/working 2>/dev/null || cd ..; rm -f eval_results.zip; zip -qr eval_results.zip repo/results/eval repo/results/eval_pilot repo/results/tables 2>/dev/null; ls -la eval_results.zip